# Player Similarity Engine — Premier League 2020/21

Finds the most similar players to any target using a blend of:
- Real performance (`xG/90`, `xA/90`, goals, assists, shots, key passes)
- FIFA attributes (pace, shooting, passing, dribbling, defending, physic)
- Physical profile (age, height in cm)
- Valuation (`value_eur`)

Original script reorganized into notebook cells without changing the core code.

In [1]:
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

## Load data

In [4]:
data = []
with open("/content/unified_players.json", "r") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"Loaded {len(data)} players")

Loaded 524 players


## Feature selection

In [5]:
# Performance features (real output)
PERF_FEATURES = ["xG_per_90", "xA_per_90", "goals", "assists", "shots", "key_passes", "npxG", "xGChain", "xGBuildup"]

# FIFA attribute features (scouting profile)
FIFA_FEATURES = ["fifa_pace", "fifa_shooting", "fifa_passing", "fifa_dribbling", "fifa_defending", "fifa_physic",
                 "attacking_finishing", "mentality_composure", "mentality_positioning", "power_shot_power"]

# Profile features
PROFILE_FEATURES = ["age", "height_cm", "value_eur"]

ALL_FEATURES = PERF_FEATURES + FIFA_FEATURES + PROFILE_FEATURES

## Filter to players with enough data
Need:
- minutes >= 450
- all selected FIFA / performance / profile fields present

In [6]:
valid_players = []
for p in data:
    if p.get("minutes", 0) and p["minutes"] >= 450:
        has_all = True
        for f in ALL_FEATURES:
            v = p.get(f)
            if v is None or v == "" or (isinstance(v, float) and np.isnan(v)):
                has_all = False
                break
        if has_all:
            valid_players.append(p)

print(f"Players with sufficient data (450+ min + all features): {len(valid_players)}")

Players with sufficient data (450+ min + all features): 315


## Build feature matrix

In [7]:
names = [p["player_name"] for p in valid_players]
teams = [p["team"] for p in valid_players]
positions = [p.get("position_us", "?") for p in valid_players]

X = np.array([[float(p[f]) for f in ALL_FEATURES] for p in valid_players])

## Normalize features and compute similarity matrix

In [8]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

sim_matrix = cosine_similarity(X_scaled)

## Lookup function

In [9]:
def find_similar(player_name, top_n=5, exclude_same_team=False):
    """Find the top_n most similar players to the given player."""
    # Fuzzy search for player name
    matches = [(i, n) for i, n in enumerate(names) if player_name.lower() in n.lower()]
    if not matches:
        print(f"Player '{player_name}' not found.")
        return None, []

    idx = matches[0][0]
    target = valid_players[idx]

    # Get similarity scores
    scores = sim_matrix[idx]

    # Sort by similarity (descending), skip self
    ranked = sorted(enumerate(scores), key=lambda x: -x[1])

    results = []
    for i, score in ranked:
        if i == idx:
            continue
        if exclude_same_team and teams[i] == teams[idx]:
            continue
        results.append({
            "player": names[i],
            "team": teams[i],
            "position": positions[i],
            "similarity": round(score * 100, 1),
            "overall": valid_players[i].get("overall"),
            "value_eur": valid_players[i].get("value_eur"),
            "xG_per_90": valid_players[i].get("xG_per_90"),
            "xA_per_90": valid_players[i].get("xA_per_90"),
        })
        if len(results) >= top_n:
            break

    return target, results

## Radar chart function

In [10]:
def make_radar(target, similars, filename):
    """Create a radar chart comparing the target player to their top similar players."""
    # Radar features (human-readable subset)
    radar_features = ["fifa_pace", "fifa_shooting", "fifa_passing", "fifa_dribbling", "fifa_defending", "fifa_physic"]
    radar_labels = ["Pace", "Shooting", "Passing", "Dribbling", "Defending", "Physical"]

    BG = "#0f1117"
    CARD = "#1a1d27"
    TEXT = "#e0e0e0"
    COLORS = ["#4fc3f7", "#ff8a65", "#81c784", "#ce93d8", "#fff176"]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection="polar"))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(CARD)

    angles = np.linspace(0, 2 * np.pi, len(radar_features), endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    # Plot target player
    target_vals = [float(target.get(f, 0)) for f in radar_features]
    target_vals += target_vals[:1]
    ax.plot(angles, target_vals, "o-", linewidth=2.5, color="#f44336", markersize=6, label=target["player_name"])
    ax.fill(angles, target_vals, alpha=0.15, color="#f44336")

    # Plot top 3 similar players
    for i, s in enumerate(similars[:3]):
        p_data = [p for p in valid_players if p["player_name"] == s["player"]][0]
        vals = [float(p_data.get(f, 0)) for f in radar_features]
        vals += vals[:1]
        ax.plot(angles, vals, "o-", linewidth=1.8, color=COLORS[i], markersize=5,
                label=f"{s['player']} ({s['similarity']}%)", alpha=0.8)
        ax.fill(angles, vals, alpha=0.05, color=COLORS[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=12, color=TEXT)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80])
    ax.set_yticklabels(["20", "40", "60", "80"], fontsize=9, color="#666")
    ax.tick_params(colors=TEXT)
    ax.grid(color="#333", linewidth=0.5)
    ax.spines["polar"].set_color("#333")

    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10,
              facecolor=CARD, edgecolor="#333", labelcolor=TEXT)

    plt.title(f"Player similarity — {target['player_name']} ({target['team']})",
              fontsize=15, fontweight="bold", color=TEXT, pad=20)
    plt.tight_layout()
    plt.savefig(filename, dpi=180, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"Radar saved: {filename}")

## Output directory and example runs

In [13]:
OUT = "/content"

examples = [
    ("Patrick Bamford", True),   # #1 value pick from Q4 — find cheaper alternatives
    ("Harry Kane", True),        # Top scorer — who profiles like him?
    ("Bruno Fernandes", True),   # Elite creative midfielder
    ("Timo Werner", True),       # Wasteful — who has similar profile but better conversion?
    ("Mohamed Salah", True),     # Elite winger
]

In [14]:
for player_name, exclude_team in examples:
    target, results = find_similar(player_name, top_n=5, exclude_same_team=exclude_team)
    if target is None:
        continue

    print(f"\n{'='*70}")
    print(f"  SIMILAR TO: {target['player_name']} ({target['team']}, {target.get('position_us','?')})")
    print(f"  FIFA: {target.get('overall')}  |  Value: €{target.get('value_eur',0)/1e6:.1f}M  |  xG/90: {target.get('xG_per_90')}  |  xA/90: {target.get('xA_per_90')}")
    print(f"{'='*70}")

    for i, r in enumerate(results, 1):
        val_str = f"€{r['value_eur']/1e6:.1f}M" if r['value_eur'] else "N/A"
        print(f"  {i}. {r['player']:25s} {r['team']:22s} {r['position']:5s}  "
              f"Sim: {r['similarity']:5.1f}%  |  FIFA: {r['overall']}  |  {val_str}  |  "
              f"xG/90: {r['xG_per_90']}  xA/90: {r['xA_per_90']}")

    # Generate radar chart
    safe_name = target["player_name"].replace(" ", "_").lower()
    make_radar(target, results, f"{OUT}/similarity_{safe_name}.png")

print(f"\n{'='*70}")
print("All similarity reports generated.")
print(f"{'='*70}")


  SIMILAR TO: Patrick Bamford (Leeds, F S)
  FIFA: 71.0  |  Value: €2.8M  |  xG/90: 0.537  |  xA/90: 0.11
  1. Ollie Watkins             Aston Villa            F      Sim:  94.2%  |  FIFA: 76.0  |  €10.5M  |  xG/90: 0.44  xA/90: 0.144
  2. Dominic Calvert-Lewin     Everton                F S    Sim:  87.3%  |  FIFA: 79.0  |  €17.0M  |  xG/90: 0.569  xA/90: 0.066
  3. Che Adams                 Southampton            F S    Sim:  86.9%  |  FIFA: 74.0  |  €7.5M  |  xG/90: 0.406  xA/90: 0.175
  4. Chris Wood                Burnley                F S    Sim:  86.4%  |  FIFA: 78.0  |  €10.5M  |  xG/90: 0.422  xA/90: 0.066
  5. Callum Wilson             Newcastle United       F S    Sim:  85.2%  |  FIFA: 78.0  |  €10.5M  |  xG/90: 0.588  xA/90: 0.132
Radar saved: /content/similarity_patrick_bamford.png

  SIMILAR TO: Harry Kane (Tottenham, F)
  FIFA: 88.0  |  Value: €71.0M  |  xG/90: 0.644  |  xA/90: 0.22
  1. Marcus Rashford           Manchester United      F M S  Sim:  90.3%  |  FIFA: 85.0

## Manual test
Use this cell to search for a single player interactively.

In [ ]:
target, results = find_similar("Harry Kane", top_n=5, exclude_same_team=True)
target, results